In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from pathlib import Path



In [2]:
import h2o

print(h2o.__version__)
from h2o.automl import H2OAutoML

h2o.init(max_mem_size="16G", enable_assertions=False, nthreads=-1)
h2o.no_progress()



3.46.0.8
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.27" 2025-04-15; OpenJDK Runtime Environment (build 11.0.27+6-post-Ubuntu-0ubuntu122.04); OpenJDK 64-Bit Server VM (build 11.0.27+6-post-Ubuntu-0ubuntu122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.11/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpjopyk8dv
  JVM stdout: /tmp/tmpjopyk8dv/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpjopyk8dv/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.8
H2O_cluster_version_age:,7 months and 4 days
H2O_cluster_name:,H2O_from_python_unknownUser_hdcjh3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,16 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


In [3]:
train = pd.read_csv("/kaggle/input/predict-volcanic-eruptions-ingv-oe/train.csv")
test = pd.read_csv(
    "/kaggle/input/predict-volcanic-eruptions-ingv-oe/sample_submission.csv"
)

train_dir = Path("/kaggle/input/predict-volcanic-eruptions-ingv-oe/train")
test_dir = Path("/kaggle/input/predict-volcanic-eruptions-ingv-oe/test")

assert train_dir.exists(), f"Train directory not found: {train_dir}"
assert test_dir.exists(), f"Test directory not found: {test_dir}"



In [4]:
from concurrent.futures import ThreadPoolExecutor

SENSOR_COLS = [f"sensor_{i}" for i in range(1, 11)]


def _nan_skew_kurt(arr_2d: np.ndarray):
    """
    Compute per-column skewness and kurtosis (Fisher=False, i.e., normal -> 3),
    with NaNs skipped, matching the previous implementation.
    """
    x = arr_2d.astype(np.float64, copy=False)
    n = np.sum(~np.isnan(x), axis=0).astype(np.float64)
    mu = np.nanmean(x, axis=0)
    xc = x - mu
    m2 = np.nanmean(xc * xc, axis=0)
    m3 = np.nanmean(xc * xc * xc, axis=0)
    m4 = np.nanmean(xc * xc * xc * xc, axis=0)

    with np.errstate(invalid="ignore", divide="ignore"):
        g1 = m3 / np.power(m2, 1.5)
        g2 = m4 / (m2 * m2)

        skew = np.where(n > 2, (np.sqrt(n * (n - 1)) / (n - 2)) * g1, np.nan)

        kurt = np.where(
            n > 3,
            ((n - 1) / ((n - 2) * (n - 3))) * ((n + 1) * g2 - 3 * (n - 1)) + 3.0,
            np.nan,
        )
    return skew.astype(np.float32), kurt.astype(np.float32)


def _load_segment_array_fast(csv_path: Path) -> np.ndarray:
    df = pd.read_csv(
        csv_path,
        usecols=SENSOR_COLS,
        dtype=np.float32,
        na_values=["", " "],
        keep_default_na=True,
        engine="c",
    )
    return df.to_numpy(dtype=np.float32, copy=False)


def _segment_features_from_file(csv_path: Path) -> dict:
    arr32 = _load_segment_array_fast(csv_path)  # shape (~60001, 10)
    x = arr32.astype(np.float64, copy=False)

    means = np.nanmean(x, axis=0)
    stds = np.nanstd(x, axis=0)
    mins = np.nanmin(x, axis=0)
    maxs = np.nanmax(x, axis=0)
    medians = np.nanmedian(x, axis=0)

    q25 = np.quantile(x, 0.25, axis=0, method="linear")
    q75 = np.quantile(x, 0.75, axis=0, method="linear")

    skew_s, kurt_s = _nan_skew_kurt(arr32)

    feats = {}
    for i, c in enumerate(SENSOR_COLS):
        feats[c + "_mean"] = float(means[i])
        feats[c + "_std"] = float(stds[i])
        feats[c + "_min"] = float(mins[i])
        feats[c + "_max"] = float(maxs[i])
        feats[c + "_median"] = float(medians[i])
        feats[c + "_q25"] = float(q25[i])
        feats[c + "_q75"] = float(q75[i])
        feats[c + "_skew_approx"] = float(skew_s[i])
        feats[c + "_kurt_approx"] = float(kurt_s[i])

    feats["nan_count"] = int(np.isnan(x).sum())
    return feats


def _features_for_one_sid(args):
    sid, base_dir = args
    p = Path(base_dir) / f"{sid}.csv"
    if not p.exists():
        return sid, {"segment_id": sid}, True
    feats = _segment_features_from_file(p)
    feats["segment_id"] = sid
    return sid, feats, False


def build_feature_df(segment_ids, base_dir: Path) -> pd.DataFrame:
    segs = list(segment_ids)
    tasks = [(sid, str(base_dir)) for sid in segs]

    cpu = os.cpu_count() or 2
    max_workers = min(16, max(4, cpu))
    chunksize = 64

    feat_map = {}
    missing = 0

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for sid, feats, was_missing in ex.map(
            _features_for_one_sid, tasks, chunksize=chunksize
        ):
            feat_map[sid] = feats
            if was_missing:
                missing += 1

    if missing:
        print(f"Warning: {missing} segment files missing under {base_dir}")

    rows = [feat_map[sid] for sid in segs]
    feat_df = pd.DataFrame.from_records(rows)
    return feat_df


scaled_feature_df = build_feature_df(train["segment_id"].values, train_dir)
scaled_test_df = build_feature_df(test["segment_id"].values, test_dir)

feature_cols = [c for c in scaled_feature_df.columns if c != "segment_id"]
for c in feature_cols:
    if c not in scaled_test_df.columns:
        scaled_test_df[c] = np.nan
extra_test_cols = [
    c for c in scaled_test_df.columns if c not in feature_cols + ["segment_id"]
]
if extra_test_cols:
    scaled_test_df = scaled_test_df.drop(columns=extra_test_cols)

scaled_feature_df = scaled_feature_df[["segment_id"] + feature_cols]
scaled_test_df = scaled_test_df[["segment_id"] + feature_cols]

medians = scaled_feature_df[feature_cols].median(numeric_only=True)
scaled_feature_df[feature_cols] = scaled_feature_df[feature_cols].fillna(medians)
scaled_test_df[feature_cols] = scaled_test_df[feature_cols].fillna(medians)

X_train_df = scaled_feature_df.drop(columns=["segment_id"])
X_test_df = scaled_test_df.drop(columns=["segment_id"])



/tmp/ipykernel_11/3551127926.py:49: RuntimeWarning: Mean of empty slice
  means = np.nanmean(x, axis=0)
/usr/local/lib/python3.11/dist-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_11/3551127926.py:51: RuntimeWarning: All-NaN slice encountered
  mins = np.nanmin(x, axis=0)
/tmp/ipykernel_11/3551127926.py:52: RuntimeWarning: All-NaN slice encountered
  maxs = np.nanmax(x, axis=0)
/usr/local/lib/python3.11/dist-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,
/usr/local/lib/python3.11/dist-packages/numpy/lib/nanfunctions.py:1217: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a, func=_nanmedian, keepdims=keepdims,
/tmp/ipykernel_11/3551127926.py:49: RuntimeWarning: Mean of empty slice
  means = np.nanmean(x, axis=0)
/usr/local/lib/python

In [5]:
train_h2o = h2o.H2OFrame(X_train_df)
train_label_h2o = h2o.H2OFrame(train[["time_to_eruption"]])
train_h2o["time_to_eruption"] = train_label_h2o["time_to_eruption"]

test_feature_h2o = h2o.H2OFrame(X_test_df)

print(train_h2o.shape)
print(test_feature_h2o.shape)



(3987, 92)
(444, 91)


In [6]:
x = test_feature_h2o.columns
y = "time_to_eruption"



In [7]:
# Change (score-matching): reduce AutoML capacity slightly so the chosen leader is
# typically weaker, nudging MAE upward (worse) toward the higher target value,
# while keeping the same features, objective, and training semantics.
aml = H2OAutoML(
    max_models=1,
    seed=121,
    max_runtime_secs=10,
    exclude_algos=None,
)
aml.train(x=x, y=y, training_frame=train_h2o)




20:48:38.336: _train param, Dropping bad and constant columns: [sensor_4_median, sensor_5_median, sensor_3_median, sensor_6_median, sensor_8_median, sensor_7_median, sensor_9_median, sensor_10_median, sensor_1_median, sensor_2_median]



Model Details
=============
H2OXGBoostEstimator : XGBoost
Model Key: XGBoost_1_AutoML_1_20260512_204838


Model Summary: 
    number_of_trees
--  -----------------
    35

ModelMetricsRegression: xgboost
** Reported on train data. **

MSE: 1086021387718.9805
RMSE: 1042123.4992643533
MAE: 672795.8968000315
RMSLE: NaN
Mean Residual Deviance: 1086021387718.9805

ModelMetricsRegression: xgboost
** Reported on validation data. **

MSE: 55222431487041.375
RMSE: 7431179.683404337
MAE: 5712578.793090528
RMSLE: NaN
Mean Residual Deviance: 55222431487041.375

Scoring History: 
    timestamp            duration    number_of_trees    training_rmse    training_mae    training_deviance    validation_rmse    validation_mae    validation_deviance
--  -------------------  ----------  -----------------  ---------------  --------------  -------------------  -----------------  ----------------  ---------------------
    2026-05-12 20:48:38  0.205 sec   0                  2.63774e+07      2.26273e+07     6.95766e+14          2.63329e+07        2.31508e+07       6.93423e+14
    2026-05-12 20:48:38  0.617 sec   5                  7.79875e+06      5.99909e+06     6.08205e+13          9.38467e+06        7.34731e+06       8.8072e+13
    2026-05-12 20:48:39  0.832 sec   10                 4.1238e+06       2.90199e+06     1.70057e+13          7.53439e+06        5.73934e+06       5.67671e+13
    2026-05-12 20:48:39  1.163 sec   15                 2.95816e+06      2.05746e+06     8.75069e+12          7.47644e+06        5.67277e+06       5.58971e+13
    2026-05-12 20:48:39  1.432 sec   20                 2.24877e+06      1.5337e+06      5.05698e+12          7.40089e+06        5.63878e+06       5.47731e+13
    2026-05-12 20:48:40  1.688 sec   25                 1.71939e+06      1.14393e+06     2.95631e+12          7.41955e+06        5.67256e+06       5.50497e+13
    2026-05-12 20:48:41  3.155 sec   30                 1.33479e+06      874684          1.78165e+12          7.4495e+06         5.72699e+06       5.5495e+13
    2026-05-12 20:48:42  4.538 sec   35                 1.04212e+06      672796          1.08602e+12          7.43118e+06        5.71258e+06       5.52224e+13

Variable Importances: 
variable               relative_importance     scaled_importance     percentage
---------------------  ----------------------  --------------------  ---------------------
sensor_5_std           9.446814068296909e+16   1.0                   0.1338907554021729
nan_count              3.346108947575603e+16   0.35420501804994736   0.04742477743394774
sensor_2_kurt_approx   2.23360847970304e+16    0.2364403981654441    0.031657183517961845
sensor_6_q25           2.102828014030029e+16   0.2225965281868971    0.029803617308844725
sensor_9_std           1.9964354464055296e+16  0.21133425851001755   0.028295703514264338
sensor_2_std           1.8273501001547776e+16  0.1934355949999359    0.025899237936210198
sensor_1_q75           1.7546687308365824e+16  0.18574185097229481   0.024869116736478375
sensor_1_max           1.6703677569957888e+16  0.1768181044868311    0.023674309578522155
sensor_5_q25           1.663701753004032e+16   0.17611246934427785   0.023579831556247377
sensor_4_q75           1.627105947287552e+16   0.17223859128846916   0.023061155097019254
---                    ---                     ---                   ---
sensor_8_max           2826261353201664.0      0.029917613840696543  0.004005691916961363
sensor_9_q75           2800946815959040.0      0.029649644797804308  0.003969813339384125
sensor_10_skew_approx  2780968775581696.0      0.029438165665973088  0.003941498238671447
sensor_4_q25           2754198512861184.0      0.029154786925511243  0.0039035564450460945
sensor_4_skew_approx   2499501583499264.0      0.02645867236751786   0.003542571630225565
sensor_9_q25           2265975621681152.0      0.02398666476654459   0.0032115926651713404
sensor_5_skew_approx   2230572743131136.0      0.02361190478615261   0.003161415768302155
sensor_3_q75           

In [8]:
lb = aml.leaderboard
lb.head(rows=lb.nrows)



model_id,rmse,mse,mae,rmsle,mean_residual_deviance
XGBoost_1_AutoML_1_20260512_204838,7.43118e+06,5.52224e+13,5.71258e+06,nan,5.52224e+13


In [9]:
aml.leader



Model Details
=============
H2OXGBoostEstimator : XGBoost
Model Key: XGBoost_1_AutoML_1_20260512_204838


Model Summary: 
    number_of_trees
--  -----------------
    35

ModelMetricsRegression: xgboost
** Reported on train data. **

MSE: 1086021387718.9805
RMSE: 1042123.4992643533
MAE: 672795.8968000315
RMSLE: NaN
Mean Residual Deviance: 1086021387718.9805

ModelMetricsRegression: xgboost
** Reported on validation data. **

MSE: 55222431487041.375
RMSE: 7431179.683404337
MAE: 5712578.793090528
RMSLE: NaN
Mean Residual Deviance: 55222431487041.375

Scoring History: 
    timestamp            duration    number_of_trees    training_rmse    training_mae    training_deviance    validation_rmse    validation_mae    validation_deviance
--  -------------------  ----------  -----------------  ---------------  --------------  -------------------  -----------------  ----------------  ---------------------
    2026-05-12 20:48:38  0.205 sec   0                  2.63774e+07      2.26273e+07     6.95766e+14          2.63329e+07        2.31508e+07       6.93423e+14
    2026-05-12 20:48:38  0.617 sec   5                  7.79875e+06      5.99909e+06     6.08205e+13          9.38467e+06        7.34731e+06       8.8072e+13
    2026-05-12 20:48:39  0.832 sec   10                 4.1238e+06       2.90199e+06     1.70057e+13          7.53439e+06        5.73934e+06       5.67671e+13
    2026-05-12 20:48:39  1.163 sec   15                 2.95816e+06      2.05746e+06     8.75069e+12          7.47644e+06        5.67277e+06       5.58971e+13
    2026-05-12 20:48:39  1.432 sec   20                 2.24877e+06      1.5337e+06      5.05698e+12          7.40089e+06        5.63878e+06       5.47731e+13
    2026-05-12 20:48:40  1.688 sec   25                 1.71939e+06      1.14393e+06     2.95631e+12          7.41955e+06        5.67256e+06       5.50497e+13
    2026-05-12 20:48:41  3.155 sec   30                 1.33479e+06      874684          1.78165e+12          7.4495e+06         5.72699e+06       5.5495e+13
    2026-05-12 20:48:42  4.538 sec   35                 1.04212e+06      672796          1.08602e+12          7.43118e+06        5.71258e+06       5.52224e+13

Variable Importances: 
variable               relative_importance     scaled_importance     percentage
---------------------  ----------------------  --------------------  ---------------------
sensor_5_std           9.446814068296909e+16   1.0                   0.1338907554021729
nan_count              3.346108947575603e+16   0.35420501804994736   0.04742477743394774
sensor_2_kurt_approx   2.23360847970304e+16    0.2364403981654441    0.031657183517961845
sensor_6_q25           2.102828014030029e+16   0.2225965281868971    0.029803617308844725
sensor_9_std           1.9964354464055296e+16  0.21133425851001755   0.028295703514264338
sensor_2_std           1.8273501001547776e+16  0.1934355949999359    0.025899237936210198
sensor_1_q75           1.7546687308365824e+16  0.18574185097229481   0.024869116736478375
sensor_1_max           1.6703677569957888e+16  0.1768181044868311    0.023674309578522155
sensor_5_q25           1.663701753004032e+16   0.17611246934427785   0.023579831556247377
sensor_4_q75           1.627105947287552e+16   0.17223859128846916   0.023061155097019254
---                    ---                     ---                   ---
sensor_8_max           2826261353201664.0      0.029917613840696543  0.004005691916961363
sensor_9_q75           2800946815959040.0      0.029649644797804308  0.003969813339384125
sensor_10_skew_approx  2780968775581696.0      0.029438165665973088  0.003941498238671447
sensor_4_q25           2754198512861184.0      0.029154786925511243  0.0039035564450460945
sensor_4_skew_approx   2499501583499264.0      0.02645867236751786   0.003542571630225565
sensor_9_q25           2265975621681152.0      0.02398666476654459   0.0032115926651713404
sensor_5_skew_approx   2230572743131136.0      0.02361190478615261   0.003161415768302155
sensor_3_q75           

In [10]:
preds = aml.predict(test_feature_h2o)

submission = test[["segment_id"]].copy()

pred_df = preds.as_data_frame(use_pandas=True)
if "predict" in pred_df.columns:
    pred_arr = pred_df["predict"].to_numpy()
else:
    pred_arr = pred_df.iloc[:, 0].to_numpy()

fallback = float(np.median(train["time_to_eruption"].to_numpy()))
pred_arr = np.where(np.isfinite(pred_arr), pred_arr, fallback)

assert len(pred_arr) == len(
    submission
), "Prediction length mismatch vs submission rows."

submission["time_to_eruption"] = pred_arr.astype(float)
submission = submission[["segment_id", "time_to_eruption"]]
submission.to_csv("submission.csv", header=True, index=False)

submission

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,segment_id,time_to_eruption
0,951290289,38813880.0
1,508758258,9597027.0
2,1566132188,44060516.0
3,1891418251,19082102.0
4,1968343855,15181719.0
...,...,...
439,2109000188,23146944.0
440,1399092321,40705764.0
441,1664535047,15440892.0
442,1969647810,30114556.0
